# Inference Pipeline

In [1]:
# Standard Libraries

import os
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# Data Manipulation

import numpy as np
import pandas as pd


# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")

# Machine Learning Models

import tensorflow as tf
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Evaluation


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.metrics import (
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# TensorFlow

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

# 15.1 Load Everything Back

A deployment notebook should always verify that saved artifacts can be reloaded.


In [4]:
# Project Configuration

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
FINAL_THRESHOLD = 0.216629

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

FIGSIZE = (12,6)

print("TensorFlow Version :", tf.__version__)
print("NumPy Version      :", np.__version__)
print("Pandas Version     :", pd.__version__)

TensorFlow Version : 2.21.0
NumPy Version      : 2.4.6
Pandas Version     : 3.0.5


In [9]:
from pathlib import Path
import joblib
from tensorflow.keras.models import load_model


DATA_DIR = Path("data/Training_Datasets")
RESULT_DIR = Path('Results')
CONFIG_DIR = Path("artifacts")

# X_train_tree = pd.read_csv(DATA_DIR / "X_train_tree.csv")
# y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")

X_test_tree  = pd.read_csv(DATA_DIR / "X_test_tree.csv")
y_test  = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

MODEL_DIR = Path("models")
lgbm_model = joblib.load(MODEL_DIR / "lightgbm.joblib")

# all_val_results = pd.read_csv(RESULT_DIR/"all_models_results.csv")


# 15.6 Build a Reusable Prediction Function

This function accepts already preprocessed features (matching the training schema).



In [6]:
def predict_transaction(
    transaction_df,
    model,
    threshold
):
    """
    Predict fraud for preprocessed transaction data.

    Parameters
    ----------
    transaction_df : pandas.DataFrame
        Preprocessed transaction(s).
    model : trained LightGBM model
    threshold : float

    Returns
    -------
    pandas.DataFrame
    """

    probability = model.predict_proba(
        transaction_df
    )[:, 1]

    prediction = (
        probability >= threshold
    ).astype(int)

    return pd.DataFrame({

        "Fraud Probability": probability,

        "Prediction": prediction

    })

# 15.7 Example Inference

In [12]:
sample = X_test_tree.iloc[[0]]

prediction = predict_transaction(

    transaction_df=sample,

    model=lgbm_model,

    threshold=FINAL_THRESHOLD

)

display(prediction)

,Fraud Probability,Prediction
0,0.000558,0


# 15.8 Batch Inference

In [14]:
batch_predictions = predict_transaction(

    transaction_df=X_test_tree.iloc[:20],

    model=lgbm_model,

    threshold=FINAL_THRESHOLD

)

display(batch_predictions)

,Fraud Probability,Prediction
0,0.000558,0
1,0.006724,0
2,0.014646,0
3,0.000616,0
4,0.002626,0
5,0.003110,0
6,0.504511,1
7,0.000723,0
8,0.238138,1
9,0.000795,0
